In [2]:
# import packages
import cv2
import glob
import numpy as np
import rawpy
import gc
import os
import psutil

# lens correction helpers
import json
import shutil
import subprocess
import lensfunpy


In [3]:
# set params

# set filepaths
use_external_drive = True
external_raw_data_dir = "/Volumes/ECM_backup/CPL26_bubbles/2502"
home_raw_data_dir = "raw_data"
raw_data_dir = external_raw_data_dir if use_external_drive else home_raw_data_dir
output_dir = "image_outputs_lens-corrected/"

# confidence threshold and scaling - note that you might need to tune these to avoid things blowing up!!!
# maybe try starting with 1 and adjusting from there?
scan_conf_thresh_lowres = 1
scan_conf_thresh_highres = 1
scan_min_inclusion_ratio = 0.90
scale_factor = 0.20  # Downsample to 20% size for feature matching


In [4]:
# Force OpenCV to run entirely on the CPU and completely disable OpenCL acceleration
# addresses an issue I was having (thanks Gemini)
cv2.ocl.setUseOpenCL(False)

# set up memory logging
process = psutil.Process(os.getpid())

def log_memory(stage):
    rss_mb = process.memory_info().rss / (1024 ** 2)
    print(f"  [Memory] {stage}: {rss_mb:.1f} MB")

['ALHIC2502-102-A', 'ALHIC2502-102-D', 'ALHIC2502-103', 'ALHIC2502-24', 'ALHIC2502-47', 'ALHIC2502-86']


In [5]:
# set params


# set filepaths
raw_data_dir = "raw_data/"
output_dir = "image_outputs/"

# confidence threshold and scaling - note that you might need to tune these to avoid things blowing up!!!
pano_conf_thresh_lowres = .85
pano_conf_thresh_highres = .9
scan_min_inclusion_ratio = 0.90
number_of_attempts = 6
scale_factor = 0.15  # Downsample to 15% size for feature matching

# lens correction
lens_correction = True


In [6]:
# Optional fallback for older Nikon metadata that may omit LensModel.
# Fill this mapping if profile lookup fails with LensID-only files.
LENS_ID_TO_MODEL = {
    # "32 54 6A 6A 24 24 35 02": "AF-S VR Micro-Nikkor 105mm f/2.8G IF-ED",
}

EXIFTOOL_AVAILABLE = shutil.which("exiftool") is not None
if not EXIFTOOL_AVAILABLE:
    print("exiftool is not installed. Install with: brew install exiftool")

LENS_DB = lensfunpy.Database() if lensfunpy is not None else None
lens_correction_ready = EXIFTOOL_AVAILABLE and (LENS_DB is not None)

LENS_META_CACHE = {}
LENS_PROFILE_CACHE = {}
LENS_MAP_CACHE = {}

def _to_float(v, default):
    try:
        return float(v)
    except Exception:
        return float(default)

def read_nef_meta(path):
    if path in LENS_META_CACHE:
        return LENS_META_CACHE[path]

    tags = [
        "Make", "Model", "LensModel", "Lens", "LensID",
        "FocalLength", "FNumber", "FocusDistance"
    ]
    cmd = ["exiftool", "-j", "-n"] + [f"-{t}" for t in tags] + [str(path)]
    out = subprocess.check_output(cmd, text=True)
    d = json.loads(out)[0]

    lens_model = d.get("LensModel") or d.get("Lens")
    if not lens_model:
        print(f"  [Info] Using fallback lens model for {path}")
        lens_model = LENS_ID_TO_MODEL.get(d.get("LensID"))

    meta = {
        "make": d.get("Make"),
        "model": d.get("Model"),
        "lens_model": lens_model,
        "lens_id": d.get("LensID"),
        "focal": _to_float(d.get("FocalLength", 50.0), 50.0),
        "aperture": _to_float(d.get("FNumber", 8.0), 8.0),
        "distance": _to_float(d.get("FocusDistance", 10.0), 10.0),
    }
    LENS_META_CACHE[path] = meta
    return meta

def _find_profile(meta):
    if LENS_DB is None:
        return None, None

    key = (meta["make"], meta["model"], meta["lens_model"])
    if key in LENS_PROFILE_CACHE:
        return LENS_PROFILE_CACHE[key]

    if not meta["make"] or not meta["model"] or not meta["lens_model"]:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    cams = LENS_DB.find_cameras(meta["make"], meta["model"], loose_search=True)
    if not cams:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None
    cam = cams[0]

    lenses = LENS_DB.find_lenses(cam, None, meta["lens_model"], loose_search=True)
    if not lenses:
        lenses = LENS_DB.find_lenses(cam, meta["make"], meta["lens_model"], loose_search=True)
    if not lenses:
        LENS_PROFILE_CACHE[key] = (None, None)
        return None, None

    lens = lenses[0]
    LENS_PROFILE_CACHE[key] = (cam, lens)
    return cam, lens

def apply_lens_correction_bgr(path, bgr):
    # No-op if dependencies are missing.
    if lensfunpy is None or shutil.which("exiftool") is None:
        return bgr

    try:
        meta = read_nef_meta(path)
    except Exception:
        return bgr

    cam, lens = _find_profile(meta)
    if cam is None or lens is None:
        return bgr

    h, w = bgr.shape[:2]
    # Cache by lens identity + image size + focal length to avoid exploding map cache.
    map_key = (
        meta["make"], meta["model"], meta["lens_model"], w, h,
        round(meta["focal"], 3)
    )

    if map_key not in LENS_MAP_CACHE:
        mod = lensfunpy.Modifier(lens, cam.crop_factor, w, h)
        mod.initialize(meta["focal"], meta["aperture"], meta["distance"])
        coords = mod.apply_geometry_distortion()
        map_x = coords[:, :, 0].astype(np.float32)
        map_y = coords[:, :, 1].astype(np.float32)
        LENS_MAP_CACHE[map_key] = (map_x, map_y)

    map_x, map_y = LENS_MAP_CACHE[map_key]
    corrected = cv2.remap(
        bgr, map_x, map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )
    return corrected

if lens_correction and not lens_correction_ready:
    print("Lens correction enabled but dependencies are missing. Running without correction.")


exiftool is not installed. Install with: brew install exiftool
Lens correction enabled but dependencies are missing. Running without correction.


In [7]:
process = psutil.Process(os.getpid())

def log_memory(stage):
    rss_mb = process.memory_info().rss / (1024 ** 2)
    print(f"  [Memory] {stage}: {rss_mb:.1f} MB")


In [8]:
# Loop through folders safely
for sample in sampleID:
    print("\n" + "="*50)
    print(f"Running images from: {sample}")
    print("="*50)

    if lens_correction and lens_correction_ready and LENS_MAP_CACHE:
        # Prevent growth of remap matrices across samples.
        LENS_MAP_CACHE.clear()
        gc.collect()

    # 1. Locate files dynamically inside the target sample folder
    nef_paths = sorted(glob.glob(f"{raw_data_dir}/{sample}/*.NEF"))
    print(f" Found {len(nef_paths)} NEF files in folder '{sample}'.")

    if len(nef_paths) == 0:
        print("  [Warning] No files found. Skipping this folder.")
        continue

    def count_integrated_images(stitcher):
        try:
            return len(stitcher.cameras())
        except Exception:
            return 0

    attempt_thresholds = []
    for attempt_idx in range(1, number_of_attempts + 1):
        if attempt_idx == 1:
            low_delta = 0.0
            high_delta = 0.0
        elif attempt_idx == 2:
            low_delta = 0.10
            high_delta = 0.05
        else:
            low_delta = 0.10 + (attempt_idx - 2) * 0.05
            high_delta = 0.05 + (attempt_idx - 2) * 0.05

        attempt_thresholds.append((
            max(0.0, pano_conf_thresh_lowres - low_delta),
            max(0.0, pano_conf_thresh_highres - high_delta)
        ))

    stitched = False
    for attempt_idx, (lowres_thresh, highres_thresh) in enumerate(attempt_thresholds, start=1):
        print(f" Attempt {attempt_idx}/{number_of_attempts} with lowres={lowres_thresh:.2f}, highres={highres_thresh:.2f}")

        # Step 1: Extract low-res thumbnails to calculate alignment geometry
        print(" Step 1: Extracting low-res thumbnails to calculate alignment...")
        low_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                rgb = raw.postprocess(use_camera_wb=True, half_size=True)

            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            if lens_correction and lens_correction_ready:
                bgr = apply_lens_correction_bgr(path, bgr)

            h, w = bgr.shape[:2]
            target_w = int(w * scale_factor * 2)
            target_h = int(h * scale_factor * 2)

            small_img = cv2.resize(bgr, (target_w, target_h), interpolation=cv2.INTER_AREA)
            low_res_images.append(np.ascontiguousarray(small_img, dtype=np.uint8))
            del rgb, bgr, small_img

        log_memory("after low-res prep")

        # Step 2: Test alignment geometry using flat grid SCANS mode
        print(" Step 2: Aligning 2D zig-zag grid structure...")
        stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher.setPanoConfidenceThresh(lowres_thresh)
        status, low_res_scan_preview = stitcher.stitch(low_res_images)
        low_res_integrated = count_integrated_images(stitcher) if status == cv2.Stitcher_OK else 0
        low_res_ratio = low_res_integrated / len(nef_paths)

        del low_res_images
        del low_res_scan_preview
        del stitcher
        gc.collect()
        log_memory("after low-res stitch")

        if status != cv2.Stitcher_OK:
            print(f"  [Error] Low-res stitching failed with error code: {status}")
            print("  Tip: Verify that the frame-to-frame step overlap is at least 20-30%.")
            if attempt_idx == len(attempt_thresholds):
                print(f"  Giving up after {number_of_attempts} attempts.")
            else:
                print("  Retrying with lower confidence thresholds...")
            continue

        print(f"  Low-res stitch included {low_res_integrated}/{len(nef_paths)} images ({low_res_ratio:.0%}).")
        if low_res_ratio < scan_min_inclusion_ratio:
            print(f"  [Warning] Low-res inclusion below {scan_min_inclusion_ratio:.0%}; retrying with lower thresholds.")
            if attempt_idx == len(attempt_thresholds):
                print(f"  Giving up after {number_of_attempts} attempts.")
            continue

        print("  Grid layout successfully calculated!")

        print(" Step 3: Generating final high-res grid composition...")
        stitcher_high = cv2.Stitcher_create(cv2.Stitcher_SCANS)
        stitcher_high.setRegistrationResol(0.6)
        stitcher_high.setPanoConfidenceThresh(highres_thresh)
        high_res_images = []
        for path in nef_paths:
            with rawpy.imread(path) as raw:
                rgb = raw.postprocess(use_camera_wb=True, half_size=False)
            bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            if lens_correction and lens_correction_ready:
                bgr = apply_lens_correction_bgr(path, bgr)
            high_res_images.append(np.ascontiguousarray(bgr, dtype=np.uint8))
            del rgb, bgr

        log_memory("after high-res prep")
        print("  Assembling canvas matrices...")
        status, high_res_scan_result = stitcher_high.stitch(high_res_images)
        high_res_integrated = count_integrated_images(stitcher_high) if status == cv2.Stitcher_OK else 0
        high_res_ratio = high_res_integrated / len(nef_paths)

        del high_res_images
        gc.collect()
        log_memory("after high-res stitch")

        if status == cv2.Stitcher_OK and high_res_ratio >= scan_min_inclusion_ratio:
            print(f"  High-res stitch included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
            output_suffix = "_FLAG" if attempt_idx > 1 else ""
            output_path = f"{output_dir}/{sample}_stitched_output{output_suffix}.jpg"
            cv2.imwrite(output_path, high_res_scan_result)
            print(f"  Success! Stitched grid saved to: {output_path}")

            del high_res_scan_result
            del stitcher_high
            gc.collect()
            stitched = True
            break

        print(f"  [Warning] High-res stitch only included {high_res_integrated}/{len(nef_paths)} images ({high_res_ratio:.0%}).")
        print("  Retrying with lower confidence thresholds...")
        if status == cv2.Stitcher_OK:
            del high_res_scan_result
        del stitcher_high
        gc.collect()
        log_memory("after failed output cleanup")

    if not stitched:
        print(f"  [Error] Could not achieve a stitch that included at least 90% of the sub-images after {number_of_attempts} attempts.")



Running images from: ALHIC2502-47
 Found 41 NEF files in folder 'ALHIC2502-47'.
 Attempt 1/6 with lowres=0.85, highres=0.90
 Step 1: Extracting low-res thumbnails to calculate alignment...
  [Memory] after low-res prep: 269.4 MB
 Step 2: Aligning 2D zig-zag grid structure...
  [Memory] after low-res stitch: 391.2 MB
  Low-res stitch included 28/41 images (68%).
  [Warning] Low-res inclusion below 90%; retrying with lower thresholds.
 Attempt 2/6 with lowres=0.75, highres=0.85
 Step 1: Extracting low-res thumbnails to calculate alignment...
  [Memory] after low-res prep: 393.3 MB
 Step 2: Aligning 2D zig-zag grid structure...
  [Memory] after low-res stitch: 409.8 MB
  Low-res stitch included 28/41 images (68%).
  [Warning] Low-res inclusion below 90%; retrying with lower thresholds.
 Attempt 3/6 with lowres=0.70, highres=0.80
 Step 1: Extracting low-res thumbnails to calculate alignment...
  [Memory] after low-res prep: 409.8 MB
 Step 2: Aligning 2D zig-zag grid structure...
  [Memory